In [2]:
import os, json, random, cv2
import json
import random
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:
# === 경로 설정 ===
fire_hydrant_images_dir = "C:/Users/User/Desktop/fire_hydrant/images"  # 원본 라벨 JSON 파일들이 있는 디렉토리 경로 (사용자 수정 필요)
fire_hydrant_labels_dir = "C:/Users/User/Desktop/fire_hydrant/labels"   # 이미지 파일들이 있는 디렉토리 경로 (현재 코드에서는 사용하지 않음)
output_dir = "output_dir"   # COCO 포맷 변환 결과를 저장할 디렉토리

In [ ]:
# === 클래스 정의 (COCO 포맷 요구사항) ===
categories = [
    {"id": 1, "name": "fire hydrant"}  # 카테고리 ID는 1번, 이름은 "fire hydrant"
]

In [ ]:
# === 클래스 정의 (COCO 포맷 요구사항) ===
categories = [
    {"id": 1, "name": "fire hydrant"}  # 카테고리 ID는 1번, 이름은 "fire hydrant"
]

# === 변환 함수 정의 ===
def convert_to_coco_format(json_files, output_path):
    coco = {
        "images": [],        # 이미지 정보 (ID, 파일명, 크기 등)
        "annotations": [],   # 어노테이션 정보 (bbox, segmentation 등)
        "categories": categories,  # 클래스 목록
    }
    annotation_id = 1  # 어노테이션 ID 초기값

    # 각 JSON 파일을 순회하며 변환
    for idx, json_file in enumerate(tqdm(json_files)):
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)  # JSON 로드

        # 이미지 정보 추출
        image_id = int(data["info"]["image_id"])
        filename = data["info"]["filename"]
        width = data["info"]["width"]
        height = data["info"]["height"]

        # 이미지 항목 추가
        coco["images"].append({
            "id": image_id,
            "file_name": filename,
            "width": width,
            "height": height,
        })

        # 어노테이션 정보 변환
        for ann in data["annotations"]:
            if ann["annotation_type"] != "bbox":
                continue  # bbox가 아니면 건너뜀
            x1, y1, x2, y2 = ann["annotation_info"][0]  # 좌상단/우하단 좌표
            w = x2 - x1
            h = y2 - y1

            coco["annotations"].append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 1,  # 소화전 클래스
                "bbox": [x1, y1, w, h],
                "area": w * h,  # 면적
                "iscrowd": 0,   # 일반 객체
            })
            annotation_id += 1

    # 결과 저장
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False, indent=2)

In [ ]:
# === 모든 JSON 파일 수집 및 학습/검증 분할 ===
all_jsons = [os.path.join(fire_hydrant_labels_dir, f) for f in os.listdir(fire_hydrant_labels_dir) if f.endswith(".json")]
random.shuffle(all_jsons)  # 무작위 셔플
split_idx = int(len(all_jsons) * 0.8)  # 80% 학습, 20% 검증
fire_hydrant_train_jsons = all_jsons[:split_idx]
fire_hydrant_val_jsons = all_jsons[split_idx:]

# === 변환 실행 ===
convert_to_coco_format(fire_hydrant_train_jsons, os.path.join(output_dir, "train.json"))
convert_to_coco_format(fire_hydrant_val_jsons, os.path.join(output_dir, "val.json"))

In [ ]:
#  모델 불러오기
yolo_version = "v8"  # "v8" 또는 "v11"으로 설정

if yolo_version == "v8":
    model_path = "yolov8n.pt"  # YOLOv8 모델 경로
elif yolo_version == "v11":
    model_path = "yolov11n.pt"  # YOLOv11 모델 경로 (가정)
else:
    raise ValueError("YOLO 버전은 'v8' 또는 'v11'만 지원합니다")

model = YOLO(model_path)  # YOLO 모델 객체 생성

# 3. 예측할 이미지 경로 설정
fire_hydrant_images_dir = "sample.jpg"  # 테스트할 이미지 경로 (사용자 수정 필요)

# 4. 예측 실행
results = model.predict(source=fire_hydrant_images_dir, save=True, conf=0.5)  # confidence threshold 0.5로 설정

# 5. 결과 출력
for result in results:
    boxes = result.boxes  # 탐지된 객체들의 바운딩 박스 정보
    for box in boxes:
        cls_id = int(box.cls[0])       # 탐지된 클래스 ID
        conf = float(box.conf[0])      # 신뢰도 (confidence score)
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # 바운딩 박스 좌표
        print(f"[클래스 {cls_id}] ({x1},{y1})-({x2},{y2}) | 신뢰도: {conf:.2f}")

In [ ]:
# 6. 이미지 시각화 (선택사항)
img = cv2.imread(fire_hydrant_images_dir)  # 이미지 로드
for result in results:
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)  # 초록색 박스
cv2.imshow("Detection", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# === 변수 설명 ===
# model_path: 사용할 YOLO 가중치 파일 경로
# fire_hydrant_images_dir: 예측할 대상 이미지 경로
# conf: 객체 탐지의 최소 신뢰도 임계값 (높을수록 엄격한 필터링)
# result.boxes: 탐지된 객체들의 바운딩 박스 정보 리스트
# box.cls: 탐지된 객체의 클래스 ID
# box.conf: 해당 탐지의 신뢰도
# box.xyxy: 바운딩 박스의 좌상단(x1, y1) ~ 우하단(x2, y2) 좌표
